In [ ]:
import numpy as np
from keras.datasets import mnist

(train_X, train_y), (test_X, test_y) = mnist.load_data()

X_train = np.array([item.flatten() for item in train_X]).T
X_test = np.array([item.flatten() for item in test_X]).T

In [ ]:
y_train = np.zeros((60000, 10))
y_train[np.arange(60000), train_y] = 1

y_test = np.zeros((10000, 10))
y_test[np.arange(10000), test_y] = 1

y_train = y_train.T
y_test = y_test.T

In [ ]:
class NeuralNets():
    """Two-layer network with manual forward pass and backpropagation."""

    def __init__(self, X, y, hidden_dim, output_dim, lr=0.001):
        self.X = X
        self.y = y
        self.lr = lr
        input_dim = self.X.shape[0]
        self.w1 = np.random.randn(hidden_dim, input_dim)
        self.w2 = np.random.randn(output_dim, hidden_dim)
        self.b1 = np.random.randn(hidden_dim)[:, None]
        self.b2 = np.random.randn(output_dim)[:, None]

    def forward_pass(self, X=None):
        if X is None:
            X = self.X

        z1 = self.w1 @ X + self.b1
        a1 = self.relu(z1)
        z2 = self.w2 @ a1 + self.b2
        self.y_hat = self.safe_softmax(z2)
        return z1, a1, z2, self.y_hat

    def relu(self, z):
        return np.where(z > 0, z, 0.01)

    def safe_softmax(self, y_pred):
        """Softmax with per-column max subtraction for numerical stability."""
        y_shift = y_pred - np.max(y_pred, axis=0)
        return np.exp(y_shift) / np.sum(np.exp(y_shift), axis=0)

    def compute_loss(self, y_actual, y_hat):
        return -np.sum(y_actual * np.log(y_hat + 1e-15)) / y_hat.shape[1]

    def backprop(self, X=None, y=None):
        """Run one mini-batch gradient descent step."""
        if X is None:
            X = self.X
        if y is None:
            y = self.y
        m = X.shape[1]

        z1, a1, z2, y_hat = self.forward_pass(X)

        dl_dz2 = self.y_hat - y
        dl_dz1 = (self.w2.T @ dl_dz2) * (a1 > 0)

        dw1 = dl_dz1 @ X.T / m
        db1 = np.sum(dl_dz1, axis=1)[:, None] / m
        dw2 = dl_dz2 @ a1.T / m
        db2 = np.sum(dl_dz2, axis=1)[:, None] / m

        self.w1 = self.w1 - self.lr * dw1
        self.w2 = self.w2 - self.lr * dw2
        self.b1 = self.b1 - self.lr * db1
        self.b2 = self.b2 - self.lr * db2

In [ ]:
def train_nn(X, y, hidden_dim=250, output_dim=10, epochs=200, lr=0.001, batch_size=10000):
    """Train with shuffled mini-batches and track the best weights."""
    nn = NeuralNets(X, y, hidden_dim, output_dim, lr)
    m = X.shape[1]
    best_accuracy = 0
    best_weights = None

    for epoch in range(epochs):
        perm = np.random.permutation(m)
        X_shuffled = X[:, perm]
        y_shuffled = y[:, perm]

        for start in range(0, m, batch_size):
            end = start + batch_size
            nn.backprop(X_shuffled[:, start:end], y_shuffled[:, start:end])

        if epoch % 10 == 0 or epoch == epochs - 1:
            nn.forward_pass(X)
            y_hat = np.argmax(nn.y_hat, axis=0)
            y_true = np.argmax(y, axis=0)
            accuracy = np.mean(y_true == y_hat)

            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_weights = (nn.w1.copy(), nn.b1.copy(), nn.w2.copy(), nn.b2.copy())

            loss = nn.compute_loss(y, nn.y_hat)
            print(f"Epoch {epoch}: train accuracy {accuracy:.4f}, loss {loss:.4f}")

    return nn, best_accuracy, best_weights


def predict(X, weights):
    """Run inference with saved weights from training."""
    w1, b1, w2, b2 = weights
    z1 = w1 @ X + b1
    a1 = np.where(z1 > 0, z1, 0.01)
    z2 = w2 @ a1 + b2
    y_shift = z2 - np.max(z2, axis=0)
    return np.exp(y_shift) / np.sum(np.exp(y_shift), axis=0)

In [ ]:
nn, train_acc, weights = train_nn(X_train, y_train)

y_hat = predict(X_test, weights)
test_acc = np.mean(np.argmax(y_hat, axis=0) == np.argmax(y_test, axis=0))
print(f"Test accuracy: {test_acc:.4f}")